# **PART 1 - SET UP THE ENVIRONMENT**

**1.1: Configure your Gemini API Key**

This notebook uses the Gemini API, which requires an API key.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Setup and authentication complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )


**1.2 Import ADK components**

In [ ]:
import os

# Kayak Link Function
from datetime import datetime, timedelta
from typing import Optional                  
from urllib.parse import quote
from typing import List, Dict, Any, Optional 

# Google ADK 
import uuid
from google.adk.agents import Agent, SequentialAgent, LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import McpToolset, google_search, AgentTool, ToolContext
from google.adk.tools.function_tool import FunctionTool
from google.adk.runners import Runner, InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.adk.apps.app import App, ResumabilityConfig
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters, Tool
from google.genai import types

print("✅ ADK components imported successfully.")



**1.3 Configue Retry Options**

When working with LLMs, you may encounter transient errors like rate limits or temporary service unavailability. Retry options automatically handle these failures by retrying the request with exponential backoff.



In [ ]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

print("✅ Retry Options Configured")

**1.4 Helper functions**

We'll define some helper functions. If you are running this outside the Kaggle environment, you don't need to do this.

In [ ]:
def show_python_code_and_result(response):
    for i in range(len(response)):
        # Check if the response contains a valid function call result from the code executor
        if (
            (response[i].content.parts)
            and (response[i].content.parts[0])
            and (response[i].content.parts[0].function_response)
            and (response[i].content.parts[0].function_response.response)
        ):
            response_code = response[i].content.parts[0].function_response.response
            if "result" in response_code and response_code["result"] != "```":
                if "tool_code" in response_code["result"]:
                    print(
                        "Generated Python Code >> ",
                        response_code["result"].replace("tool_code", ""),
                    )
                else:
                    print("Generated Python Response >> ", response_code["result"])


print("✅ Helper functions defined.")

**1.5: Set up proxy and tunneling¶**

We'll use a proxy to access the ADK web UI from within the Kaggle Notebooks environment. If you are running this outside the Kaggle environment, you don't need to do this.

In [ ]:
from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers


# Gets the proxied URL in the Kaggle Notebooks environment
def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"

    servers = list(list_running_servers())
    if not servers:
        raise Exception("No running Jupyter servers found.")

    baseURL = servers[0]["base_url"]

    try:
        path_parts = baseURL.split("/")
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError:
        raise Exception(f"Could not parse kernel/token from base URL: {baseURL}")

    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"

    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong>⚠️ IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            The ADK web UI is <strong>not running yet</strong>. You must start it in the next cell.
            <ol style="margin-top: 10px; padding-left: 20px;">
                <li style="margin-bottom: 5px;"><strong>Run the next cell</strong> (the one with <code>!adk web ...</code>) to start the ADK web UI.</li>
                <li style="margin-bottom: 5px;">Wait for that cell to show it is "Running" (it will not "complete").</li>
                <li>Once it's running, <strong>return to this button</strong> and click it to open the UI.</li>
            </ol>
            <em style="font-size: 0.9em; color: #555;">(If you click the button before running the next cell, you will get a 500 error.)</em>
        </div>
        <a href='{url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            Open ADK Web UI (after running cell below) ↗
        </a>
    </div>
    """

    display(HTML(styled_html))

    return url_prefix


print("✅ Helper functions defined.")

**2.1 Create Sub-Agent 1 - Parameter Extraction Agent**

# **PART 2 - CREATE AGENTS**

In [ ]:
FLIGHT_PARAMETERS = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "origin_code": {"type": "string", "description": "The name of the origin city."},
            "destination_code": {"type": "string", "description": "The name of the destination city."},
            "departure_date": {"type": "string", "description": "The departure date."},
            "return_date": {"type": "string", "description": "The return date."},
            "adults": {"type": "integer", "description": "The number of passengers."},
            "cabin_class": {"type": "string", "description": "The cabin class."}
        },
    }
}

parameter_extraction_agent = Agent(
    name="Parameter_Extraction_Agent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are an expert NLP parser. Your only job is to analyze the user's flight 
    request and extract the following parameters: origin_code, destination_code, departure_date 
    (YYYY-MM-DD), return_date (YYYY-MM-DD), adults (default 1), and cabin_class (default 'economy').
    Use 'google_search' tool to get the International Air Transport Association (IATA) format 
    for the origin_code and destination_code. For the origin_code and destination_code, 
    it must be in the 3-letter IATA format. 
    You must return the result in the exact JSON format matching the FlightParameters schema.""",
    tools=[google_search],
    output_key="parameter_extraction",  # The result of this agent will be stored in the session state with this key.
)

print("✅ parameter_extraction_agent created")

**2.2. Create Sub-Agent 2 - Kayak URL Agent**

In [ ]:
from dateutil import parser
from typing import Optional

def parse_date_to_iso(date_str: str) -> Optional[str]:
    """Converts common date strings (e.g., '30 December 2025') to YYYY-MM-DD format."""
    try:
        # dateutil.parser is highly robust for parsing different date formats
        return parser.parse(date_str).strftime('%Y-%m-%d')
    except (ValueError, TypeError):
        # Return the original string if parsing fails (it might already be in ISO format)
        return date_str

def create_kayak_flights_url(
    origin_code: str,
    destination_code: str,
    departure_date: str,
    return_date: Optional[str] = None,
    adults: int = 1,
    cabin_class: str = "economy",
) -> str:
    """
    Constructs a precise Kayak search URL based on user-parsed criteria, handling defaults.
    
    Baggage is handled by the 'bags' parameter, which is 0 by default.
    """

    # 1. CLEANING STEP: Ensure dates are in YYYY-MM-DD format
    departure_date = parse_date_to_iso(departure_date)
    if return_date:
        return_date = parse_date_to_iso(return_date)
    
    # 2. Ensure that the wording fits Kayak's version.
    class_map = {
        "economy": "",
        "premium": "premium",
        "business": "business",
        "first": "first"
    }
    cabin_code = class_map.get(cabin_class.lower(), "") # Default to Economy
    
    # Base Kayak URL for searches
    BASE_URL = "https://www.kayak.com/flights"
    
    # 1. Trip Type and Dates
    if return_date:
        # Round-Trip format: /ORIGIN-DESTINATION/DEPART_DATE/RETURN_DATE
        trip_segment = f"/{origin_code}-{destination_code}/{departure_date}/{return_date}"
    else:
        # One-Way format: /ORIGIN-DESTINATION/DEPART_DATE
        trip_segment = f"/{origin_code}-{destination_code}/{departure_date}"
        
    # 2. Query Parameters (pax, cabin, bags)
    if adults <= 9:
        query_params = [
            f"/{cabin_code}/", # Cabin Class
            f"{adults}adults?", # Adults/Passengers
        ]
    else:
        query_params = [
            f"/{cabin_code}/", # Cabin Class
            f"9adults?", # 9 Adults/Passenger is the maximum
        ]
        print("# 9 adults/passenger is the maximum")
    
    # 3. Final URL Assembly
    final_url = f"{BASE_URL}{trip_segment}{''.join(query_params)}"
    return final_url.replace(" ", "")

# Register the custom function as an ADK tool
google_flights_url_tool = FunctionTool( # Renaming the function tool for clarity
    func=create_kayak_flights_url,
)

print("✅ create_kayak_flights_url created")


In [ ]:
# Use the FunctionTool you defined
create_kayak_flights_url = FunctionTool(
    func=create_kayak_flights_url
)

In [ ]:
kayak_url_agent = Agent(
    name="Kayak_URL_Agent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are an expert URL constructor. Your only task is to take the flight 
    parameters provided by the 'parameter_extraction_agent'. The parameters are stored in 
    output of parameter_extraction_agent called 'parameter_extraction'. Once you received the
    'parameter_extraction', you must immediately call the 'create_kayak_flights_url' tool to 
    generate the final Kayak flight search URL. Do not reason about the data; simply call 
    the tool and return the URL.""",
    tools=[create_kayak_flights_url],
    output_key="kayak_flights_url",  # The result of this agent will be stored in the session state with this key.
)

print("✅ kayak_url_agent created")

**2.3 Create Sub-Agent 3 - BrowserBase Scrape Agent**

In [ ]:
# MCP integration with Everything Server
BROWSERBASE_API_KEY = "YOUR_BROWSERBASE_API_KEY" # Key in your Browserbase API Key
BROWSERBASE_PROJECT_ID = "YOUR_BROSWERBASE_PROJECT_ID" # Key in your Browserbase Project ID
GEMINI_API_KEY = GOOGLE_API_KEY

browserbase_mcp_tools: McpToolset = McpToolset(
            connection_params=StdioConnectionParams(
                server_params = StdioServerParameters(
                    command="npx",
                    args=[
                        "-y",
                        "@browserbasehq/mcp-server-browserbase",
                    ],
                    env={
                        "BROWSERBASE_API_KEY": BROWSERBASE_API_KEY,
                        "BROWSERBASE_PROJECT_ID": BROWSERBASE_PROJECT_ID,
                        "GEMINI_API_KEY": GEMINI_API_KEY,
                    }
                ),
                tool_filter=[
        "browserbase_session_create",      # To start the browser session
        "browserbase_stagehand_navigate", # To go to a specific URL
        "browserbase_stagehand_act",      # To click buttons, type text, etc.
        "browserbase_stagehand_extract",  # To scrape structured data
        "browserbase_session_close",      # To close the session
    ],
                timeout=30,
            ),
        )

print("✅ MCP BrowserBase Tool created")

In [ ]:
FLIGHT_DATA_SCHEMA = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "airlines": {"type": "string", "description": "The name of the airlines."},
            "stops": {"type": "string", "description": "The number of stops."},
            "duration": {"type": "string", "description": "The flight duration."},
            "price": {"type": "integer", "description": "The price of the flight tickets."},
        },
    }
}

print("✅  FLIGHT_DATA_SCHEMA created")

In [ ]:
browserbase_scrape_agent = Agent(
    name="Browserbase_Scrape_Agent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are an expert web scraping agent. Your task is to use the 
    'browserbase_session_create', 'browserbase_stagehand_navigate', and 
    'browserbase_stagehand_extract' tools from the BrowserBase MCP toolset to fetch 
    flight data.
    
    1. **Start** a new BrowserBase session.
    2. **Navigate** to the URL provided in the session state under the key 'kayak_flights_url'.
    3. **Wait** for the flight results page to fully load (look for flight listings).
    4. **Extract** the flight data (airlines, stops, duration, price) using the 
       FLIGHT_DATA_SCHEMA.
    5. **Close** the BrowserBase session.
    6. **Store** the extracted flight data as the 'output_key' called "scraped_summary"   
    
    Return the structured flight data directly.""",
    tools=[browserbase_mcp_tools],
    output_key="scraped_summary",  # The result of this agent will be stored in the session state with this key.
)

print("✅ kayak_url_agent created")


**2.4. Create Sub-Agent 4 - Summarizer Agent**

In [ ]:
# Summarizer Agent: Its job is to summarize the text it receives.
summarizer_agent = Agent(
    name="SummarizerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The instruction is modified to request a bulleted list for a clear output format.
    instruction="""Read the provided research findings: {scraped_summary}
Create a concise summary as a bulleted list with 3-5 key points.""",
    output_key="final_summary",
)

print("✅ summarizer_agent created.")

**2.5. Create Root Agent**

In [ ]:
root_agent = SequentialAgent(
    name = "BlogPipeline",
    sub_agents = [parameter_extraction_agent, 
                  kayak_url_agent, 
                  browserbase_scrape_agent, 
                  summarizer_agent]
)

print("✅ Sequential Agent created")

In [ ]:
runner = InMemoryRunner(agent = root_agent)
response = await runner.run_debug(
    "Find me the flight options from Kuala LUMPUR to Singapore leaving on 15 December 2025 and returning on 30 December 2025 for 5 people."
)

print("✅ Done")